# Text vs Image extraction comparison

Compares extracting invoice data two ways from the **same PDF**:

- **Path A (text)** -- `pdf_reader.py`'s 2D-layout-canvas text extraction, fed through the real production pipeline (`llm_extract.extract_fields`, including all its text-based guardrails).
- **Path B (image)** -- the PDF's own page(s) rendered straight to an image and sent to the model, using the *same* system prompts (`_HEADER_SYSTEM_PROMPT` / `_LINE_ITEMS_SYSTEM_PROMPT`) and the *same* model, so any difference is attributable to the input modality, not the prompt.

This is intentionally **not** a perfectly controlled A/B (Path A gets production's text-only guardrails, Path B is a raw two-pass baseline with none) -- it's "what the real pipeline does today" vs "what a naive image-only pass would get", which is the comparison that's actually useful to look at.

## Suggested hard PDF

`6810995125_Commercial Invoice_1.pdf` (in `C:\Users\T106\Downloads\Set_1\Set_1`). This one's digital text layer is genuinely corrupted **at the source** -- confirmed earlier: the Sold-To and Ship-To address columns are interleaved in the PDF's own content stream, so even raw `pdfplumber` character extraction (before any of this project's own clean-up logic runs) already reads "INDIA" as "INDIPrivate", "MOULD PLOT" as "MOOWPLOT", etc. Since the corruption is baked into the text layer itself -- not something `pdf_reader.py`'s de-dupe logic can fix -- this is exactly the kind of file where sending the raw page *image* (bypassing text extraction entirely) has the best real chance of beating the text path. It's a genuinely digital (selectable-text) PDF, not a scan, so it's a fair test of text-vs-image rather than OCR-vs-image.

Swap `PDF_PATH` below for any other selectable-text PDF to compare instead.

In [1]:
import sys, os, io, json, time, base64
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))  # this notebook lives in Pray/, project root

import pypdfium2 as pdfium
from openai import AsyncOpenAI

from pdf_reader import extract_text
from llm_extract import (
    extract_fields,
    _HEADER_SYSTEM_PROMPT,
    _LINE_ITEMS_SYSTEM_PROMPT,
    _is_reasoning_model,
    _extract_json,
    _extract_usage,
    _calculate_cost,
    _call_header_model,
    _call_line_items_model,
)
# llm_extract.py already calls load_dotenv() on import, so OPENAI_API_KEY
# from a .env file in the project root is picked up automatically.

ImportError: cannot import name 'extract_fields' from 'llm_extract' (d:\prashantproject\Pray\llm_extract.py)

In [ ]:
PDF_PATH = r"C:\Users\T106\Downloads\Set_1\Set_1\6810995125_Commercial Invoice_1.pdf"
MODEL = "gpt-5-nano"   # same model for both paths -- keeps the comparison to input modality only
IMAGE_DPI = 250          # render scale for the image path; raise if fine print gets lost

## Path A -- extracted text -> LLM

In [ ]:
text, tables_text, method, ocr_time, scan_quality = extract_text(PDF_PATH)
print(f"extraction method: {method}  ({len(text)} chars, ocr_time={ocr_time}s)")
print(text[:3000])

In [ ]:
t0 = time.perf_counter()
text_result = await extract_fields(text, model=MODEL, tables_text=tables_text)
text_elapsed = time.perf_counter() - t0

text_header = text_result["invoice_header"]
text_items = text_result["line_items"]
text_usage = text_result["token_usage"]

print(f"[TEXT PATH] {text_elapsed:.1f}s, {text_usage['total_tokens']} tokens, "
      f"${text_usage['cost_usd']:.5f} (INR {text_usage['cost_inr']:.4f})")
print(json.dumps(text_header, indent=2, ensure_ascii=False))
print(f"\n{len(text_items)} line item(s):")
print(json.dumps(text_items, indent=2, ensure_ascii=False))

## Path B -- raw page image -> LLM

In [ ]:
def render_pdf_to_data_urls(pdf_path: str, dpi: int = 250) -> list[str]:
    """Renders every page of `pdf_path` to a base64 PNG data URL -- same
    pypdfium2 rasterization call pdf_reader.py's OCR fallback uses."""
    scale = dpi / 72
    doc = pdfium.PdfDocument(pdf_path)
    urls = []
    for page in doc:
        bitmap = page.render(scale=scale)
        pil_image = bitmap.to_pil()
        buf = io.BytesIO()
        pil_image.save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode("ascii")
        urls.append(f"data:image/png;base64,{b64}")
    return urls


image_data_urls = render_pdf_to_data_urls(PDF_PATH, dpi=IMAGE_DPI)
print(f"{len(image_data_urls)} page image(s) rendered")

# quick visual sanity check
from IPython.display import Image, display
for url in image_data_urls:
    display(Image(data=base64.b64decode(url.split(',', 1)[1]), width=500))

In [ ]:
def _image_content_blocks(urls: list[str]) -> list[dict]:
    return [{"type": "image_url", "image_url": {"url": u, "detail": "high"}} for u in urls]


async def extract_header_from_image(urls: list[str], model: str = MODEL):
    client = AsyncOpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    user_content = [{"type": "text", "text": "Extract the invoice header fields from this document image."}]
    user_content += _image_content_blocks(urls)

    request_kwargs = dict(
        model=model,
        messages=[
            {"role": "system", "content": _HEADER_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
    )
    if not _is_reasoning_model(model):
        request_kwargs["response_format"] = {"type": "json_object"}

    resp = await _call_header_model(client, request_kwargs)
    usage = _extract_usage(resp)
    return _extract_json(resp.choices[0].message.content), usage


async def extract_items_from_image(urls: list[str], model: str = MODEL):
    client = AsyncOpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    user_content = [{"type": "text", "text": "Extract every line item from this document image's table."}]
    user_content += _image_content_blocks(urls)

    request_kwargs = dict(
        model=model,
        messages=[
            {"role": "system", "content": _LINE_ITEMS_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
    )
    if not _is_reasoning_model(model):
        request_kwargs["response_format"] = {"type": "json_object"}

    resp = await _call_line_items_model(client, request_kwargs)
    usage = _extract_usage(resp)
    data = _extract_json(resp.choices[0].message.content)
    if isinstance(data, list):
        return data, usage
    if isinstance(data, dict):
        for key in ("line_items", "items", "ITEM", "items_list"):
            if key in data and isinstance(data[key], list):
                return data[key], usage
    return [], usage

In [ ]:
t0 = time.perf_counter()
image_header, header_usage = await extract_header_from_image(image_data_urls, model=MODEL)
image_items, items_usage = await extract_items_from_image(image_data_urls, model=MODEL)
image_elapsed = time.perf_counter() - t0

total_input = header_usage["input_tokens"] + items_usage["input_tokens"]
total_cached = header_usage["cached_tokens"] + items_usage["cached_tokens"]
total_output = header_usage["output_tokens"] + items_usage["output_tokens"]
image_usage = _calculate_cost(MODEL, total_input, total_cached, total_output)

print(f"[IMAGE PATH] {image_elapsed:.1f}s, {image_usage['total_tokens']} tokens, "
      f"${image_usage['cost_usd']:.5f} (INR {image_usage['cost_inr']:.4f})")
print(json.dumps(image_header, indent=2, ensure_ascii=False))
print(f"\n{len(image_items)} line item(s):")
print(json.dumps(image_items, indent=2, ensure_ascii=False))

## Side-by-side comparison

In [ ]:
import pandas as pd

header_fields = sorted(set(text_header) | set(image_header))
rows = [
    {
        "field": f,
        "TEXT path": text_header.get(f),
        "IMAGE path": image_header.get(f),
        "match": text_header.get(f) == image_header.get(f),
    }
    for f in header_fields
]
pd.set_option("display.max_colwidth", 80)
pd.DataFrame(rows)

In [ ]:
pd.DataFrame([
    {"path": "TEXT", "elapsed_s": round(text_elapsed, 1), "total_tokens": text_usage["total_tokens"],
     "cost_usd": text_usage["cost_usd"], "cost_inr": text_usage["cost_inr"]},
    {"path": "IMAGE", "elapsed_s": round(image_elapsed, 1), "total_tokens": image_usage["total_tokens"],
     "cost_usd": image_usage["cost_usd"], "cost_inr": image_usage["cost_inr"]},
])

## Observations

_Fill in after running:_
- Did the image path recover `supplier_name` / `buyer_name` / addresses correctly where the text path's column-interleaving corrupted them?
- Any fields the TEXT path got right and the IMAGE path missed (e.g. small print, a value the model couldn't read at this DPI)?
- Which path was cheaper / faster, and by how much?
- If the image path wins on accuracy here, is it worth the extra image tokens on the invoices where the text path already works fine -- or is this best used as a targeted fallback specifically for files with confirmed source-level text corruption?